# How Confident Is This Prediction?

**A model that says "42 plus or minus 3" is more useful than one that just says "42."**

Most ML libraries hand you a point prediction and call it a day. But in the real world, you need to know *how much to trust* that number. Is it a rock-solid estimate backed by dense training data? Or a wild guess extrapolated into the void?

FerroML treats uncertainty as a first-class citizen. In this notebook, we'll explore:

1. **Linear Regression** with built-in prediction intervals -- no extra packages needed
2. **Gaussian Processes** that give you a full posterior distribution over predictions
3. **When the model knows it doesn't know** -- uncertainty that grows in data-sparse regions
4. **Practical decision-making** with confidence bands

## Setting the Stage: A Tricky Dataset

We'll create a 1D regression problem with an intentional **gap** in the training data. This forces the model to make predictions in a region where it has *no information* -- the perfect stress test for uncertainty estimation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# True function: a gentle sine wave on a linear trend
def true_function(x):
    return 2.0 * x + 1.5 * np.sin(x)

# Training data: clustered in two regions with a GAP in the middle
x_train_left = np.random.uniform(0, 3, 15)
x_train_right = np.random.uniform(6, 9, 15)
x_train = np.sort(np.concatenate([x_train_left, x_train_right]))
noise = np.random.normal(0, 0.5, len(x_train))
y_train = true_function(x_train) + noise

# Dense grid for predictions (including the gap and extrapolation zones)
x_plot = np.linspace(-1, 11, 300)

# Reshape for FerroML (expects 2D arrays)
X_train = x_train.reshape(-1, 1)
X_plot = x_plot.reshape(-1, 1)

print(f"Training points: {len(x_train)}")
print(f"Gap region:      x in [3, 6] -- no training data here")
print(f"Extrapolation:   x < 0 and x > 9")

## Part 1: Linear Regression with Prediction Intervals

In sklearn, `LinearRegression.predict()` gives you a single number. Want confidence intervals? You'll need to compute them yourself with residual standard errors, leverage matrices, and t-distributions.

In FerroML, it's one method call: **`predict_interval(X, level=0.95)`**.

In [ ]:
from ferroml.linear import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

# Point predictions
y_pred_lr = lr.predict(X_plot)

# Prediction intervals at two confidence levels -- one line each
preds_95, lower_95, upper_95 = lr.predict_interval(X_plot, level=0.95)
preds_80, lower_80, upper_80 = lr.predict_interval(X_plot, level=0.80)

# FerroML also gives you a full statsmodels-style summary for free
print(lr.summary())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Shaded prediction intervals -- wider = less certain
ax.fill_between(x_plot, lower_95, upper_95, alpha=0.15, color='steelblue',
                label='95% prediction interval')
ax.fill_between(x_plot, lower_80, upper_80, alpha=0.25, color='steelblue',
                label='80% prediction interval')

# Point prediction and ground truth
ax.plot(x_plot, y_pred_lr, color='steelblue', linewidth=2, label='Prediction')
ax.plot(x_plot, true_function(x_plot), 'k--', linewidth=1, alpha=0.5,
        label='True function')
ax.scatter(x_train, y_train, c='orangered', s=40, zorder=5,
           edgecolors='k', linewidth=0.5, label='Training data')

# Highlight the data gap
ax.axvspan(3, 6, alpha=0.05, color='red')
ax.text(4.5, ax.get_ylim()[0] + 1, 'No training\ndata here',
        ha='center', fontsize=10, color='darkred', style='italic')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Linear Regression: Prediction Intervals Widen Away From Data',
             fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Notice how the prediction intervals **bow outward** as we move away from the center of the training data. Linear regression knows its limits -- predictions at the edges carry more uncertainty.

But there's a catch: linear regression assumes a *linear* relationship, so it can't capture the fact that the gap region (x=3 to x=6) is fundamentally different from extrapolation. The intervals widen smoothly based on distance from the data centroid, not from the nearest data point.

For **nonlinear, data-aware uncertainty**, we need something more powerful.

## Part 2: Gaussian Processes -- The Gold Standard

A Gaussian Process doesn't just fit a line. It maintains a *probability distribution over all possible functions* that could explain the data. Where training data is dense, the posterior collapses to a tight band. Where data is absent, uncertainty balloons.

This is a model that genuinely **knows what it doesn't know**.

In [ ]:
from ferroml.gaussian_process import GaussianProcessRegressor

gp = GaussianProcessRegressor()
gp.fit(X_train, y_train)

# predict_with_std returns (mean, standard_deviation) -- the full posterior
gp_mean, gp_std = gp.predict_with_std(X_plot)

# Convert to confidence bands
gp_upper_95 = gp_mean + 1.96 * gp_std  # 95% band
gp_lower_95 = gp_mean - 1.96 * gp_std
gp_upper_68 = gp_mean + gp_std          # 68% band (1 std)
gp_lower_68 = gp_mean - gp_std

# Look at the numbers -- this tells the whole story
print(f"Uncertainty at a training point (x=1.0):  std = {gp_std[np.argmin(np.abs(x_plot - 1.0))]:.6f}")
print(f"Uncertainty in the data gap (x=4.5):      std = {gp_std[np.argmin(np.abs(x_plot - 4.5))]:.6f}")
print(f"Uncertainty at extrapolation (x=11.0):     std = {gp_std[np.argmin(np.abs(x_plot - 11.0))]:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Shaded confidence bands
ax.fill_between(x_plot, gp_lower_95, gp_upper_95, alpha=0.15, color='darkorange',
                label='95% confidence band')
ax.fill_between(x_plot, gp_lower_68, gp_upper_68, alpha=0.3, color='darkorange',
                label='68% confidence band (1 std)')

# Mean prediction and ground truth
ax.plot(x_plot, gp_mean, color='darkorange', linewidth=2, label='GP mean prediction')
ax.plot(x_plot, true_function(x_plot), 'k--', linewidth=1, alpha=0.5,
        label='True function')
ax.scatter(x_train, y_train, c='orangered', s=40, zorder=5,
           edgecolors='k', linewidth=0.5, label='Training data')

# Annotate the gap
ax.axvspan(3, 6, alpha=0.05, color='red')
ax.text(4.5, ax.get_ylim()[0] + 1, 'Uncertainty\nexplodes here',
        ha='center', fontsize=10, color='darkred', style='italic')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title("Gaussian Process: The Model Knows What It Doesn't Know",
             fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Look at the difference from linear regression:

- **Near training data**: the GP bands collapse to nearly zero width -- it's *certain* about those predictions
- **In the gap (x=3 to 6)**: uncertainty fans out dramatically, even though linear regression was perfectly happy extrapolating its line through there
- **Beyond the training range**: the GP honestly reverts to maximum uncertainty instead of confidently extending a trend

This is what principled uncertainty looks like.

## Part 3: Head-to-Head -- Linear Regression vs. Gaussian Process

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# --- Linear Regression ---
ax1.fill_between(x_plot, lower_95, upper_95, alpha=0.2, color='steelblue')
ax1.plot(x_plot, y_pred_lr, color='steelblue', linewidth=2)
ax1.plot(x_plot, true_function(x_plot), 'k--', linewidth=1, alpha=0.5)
ax1.scatter(x_train, y_train, c='orangered', s=30, zorder=5,
            edgecolors='k', linewidth=0.5)
ax1.axvspan(3, 6, alpha=0.05, color='red')
ax1.set_title('Linear Regression', fontsize=14, fontweight='bold')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('y', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.text(4.5, -3, 'Intervals widen\ngradually', ha='center',
         fontsize=9, color='steelblue', style='italic')

# --- Gaussian Process ---
ax2.fill_between(x_plot, gp_lower_95, gp_upper_95, alpha=0.2, color='darkorange')
ax2.plot(x_plot, gp_mean, color='darkorange', linewidth=2)
ax2.plot(x_plot, true_function(x_plot), 'k--', linewidth=1, alpha=0.5)
ax2.scatter(x_train, y_train, c='orangered', s=30, zorder=5,
            edgecolors='k', linewidth=0.5)
ax2.axvspan(3, 6, alpha=0.05, color='red')
ax2.set_title('Gaussian Process', fontsize=14, fontweight='bold')
ax2.set_xlabel('x', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.text(4.5, -3, 'Uncertainty explodes\nwhere data is missing', ha='center',
         fontsize=9, color='darkorange', style='italic')

fig.suptitle('Two Ways to Say "I\'m Not Sure"', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Uncertainty as a Decision-Making Tool

Uncertainty isn't just an academic exercise. In production systems, you can use it to:

- **Flag unreliable predictions** for human review
- **Route difficult cases** to a more expensive model
- **Set confidence thresholds** that match your risk tolerance

Let's see this in action with four test points -- two near training data, two in dangerous territory.

In [ ]:
# Four test points: two safe, two dangerous
x_new = np.array([1.5, 4.5, 7.5, 10.5]).reshape(-1, 1)
labels = ['Near data\n(x=1.5)', 'In the gap\n(x=4.5)',
          'Near data\n(x=7.5)', 'Extrapolation\n(x=10.5)']

gp_means, gp_stds = gp.predict_with_std(x_new)

fig, ax = plt.subplots(figsize=(10, 5))

# Green = trustworthy, Red = proceed with caution
colors = ['#2ecc71' if s < 0.3 else '#e74c3c' for s in gp_stds]
bars = ax.bar(range(4), gp_stds, color=colors, edgecolor='k',
              linewidth=0.5, alpha=0.8)

ax.set_xticks(range(4))
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Prediction uncertainty (std. dev.)', fontsize=12)
ax.set_title('Should You Trust This Prediction?', fontsize=14)

# Trust threshold line
ax.axhline(y=0.3, color='gray', linestyle='--', alpha=0.7)
ax.text(3.6, 0.33, 'Trust threshold', fontsize=10, color='gray', ha='right')

# Value labels on bars
for bar, std, mean in zip(bars, gp_stds, gp_means):
    verdict = "TRUST" if std < 0.3 else "REVIEW"
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'std={std:.3f}\n{verdict}',
            ha='center', fontsize=10, fontweight='bold')

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Part 5: The Full Picture -- Uncertainty Across the Input Space

Let's combine everything into one view. This plot shows the GP's uncertainty as a heatmap-style background, making it immediately obvious where the model is confident and where it's guessing.

In [ ]:
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(12, 8),
                                      height_ratios=[3, 1], sharex=True)

# Top: GP predictions with confidence bands
ax_top.fill_between(x_plot, gp_lower_95, gp_upper_95, alpha=0.15,
                    color='darkorange', label='95% confidence band')
ax_top.fill_between(x_plot, gp_lower_68, gp_upper_68, alpha=0.3,
                    color='darkorange', label='68% confidence band')
ax_top.plot(x_plot, gp_mean, color='darkorange', linewidth=2,
            label='GP mean')
ax_top.plot(x_plot, true_function(x_plot), 'k--', linewidth=1, alpha=0.5,
            label='True function')
ax_top.scatter(x_train, y_train, c='orangered', s=40, zorder=5,
               edgecolors='k', linewidth=0.5, label='Training data')
ax_top.set_ylabel('y', fontsize=12)
ax_top.set_title('Gaussian Process: Predictions + Uncertainty Map', fontsize=14)
ax_top.legend(loc='upper left', fontsize=10)
ax_top.grid(True, alpha=0.3)

# Bottom: Uncertainty heatmap
# Color-code by uncertainty level
ax_bot.fill_between(x_plot, 0, gp_std, alpha=0.8, color='darkorange')
ax_bot.axhline(y=0.3, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax_bot.text(10.5, 0.33, 'Trust\nthreshold', fontsize=8, color='red',
            ha='center', va='bottom')
ax_bot.set_xlabel('x', fontsize=12)
ax_bot.set_ylabel('Std. Dev.', fontsize=12)
ax_bot.set_ylim(0, max(gp_std) * 1.15)
ax_bot.grid(True, alpha=0.3)

# Mark training data regions on uncertainty plot
for xt in x_train:
    ax_bot.axvline(x=xt, color='gray', alpha=0.15, linewidth=0.5)

plt.tight_layout()
plt.show()

## Key Takeaways

| Feature | sklearn | FerroML |
|---------|---------|---------|
| Linear Regression point prediction | `model.predict(X)` | `model.predict(X)` |
| Linear Regression prediction intervals | Manual computation required | **`model.predict_interval(X, level=0.95)`** |
| Linear Regression model summary | Not available | **`model.summary()`** (statsmodels-style) |
| Gaussian Process with uncertainty | `model.predict(X, return_std=True)` | **`model.predict_with_std(X)`** |

**The bottom line:** FerroML gives you uncertainty estimation as a first-class feature, not an afterthought. A prediction without a confidence interval is just a number. A prediction *with* one is actionable intelligence.